# Unfair Clause Classification with Legal-BERT

## Setup

In [12]:
import re
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [14]:
MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

DATA_DIR = "/kaggle/input/datasets/mohmmedtamer/sic-graduation-project-bert"
SAVE_DIR = "/kaggle/working/legal-bert"

NAMES = ["Limitation of liability", "Unilateral termination", "Unilateral change", "Content removal",
         "Contract by using", "Choice of law", "Jurisdiction", "Arbitration"]

## Load data

In [15]:
def to_vector(labels):
    ids = [int(x) for x in re.findall(r"\d+", labels)]
    return [1.0 if i in ids else 0.0 for i in range(8)]

train = pd.read_csv(f"{DATA_DIR}/train.csv")
val = pd.read_csv(f"{DATA_DIR}/val.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

for df in [train, val, test]:
    df["text"] = df["text"].fillna("")
    df["labels"] = df["labels"].apply(to_vector)

y_train = np.array(train["labels"].tolist())
y_val = np.array(val["labels"].tolist())
y_test = np.array(test["labels"].tolist())

train.head()

,text,labels
0,notice to california subscribers : you may can...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
1,"if you subscribed using your apple id , refund...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
2,"if you wish to request a refund , please visit...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
3,if you subscribed using your google play store...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
4,key changes in this version : we 've included ...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"


## Baseline: TF-IDF + Logistic Regression

In [16]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X_train = tfidf.fit_transform(train["text"])
X_test = tfidf.transform(test["text"])

baseline = OneVsRestClassifier(LogisticRegression(max_iter=2000, class_weight="balanced"))
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)
print("Baseline micro F1:", round(f1_score(y_test, baseline_pred, average="micro") * 100, 1))
print("Baseline macro F1:", round(f1_score(y_test, baseline_pred, average="macro") * 100, 1))

Baseline micro F1: 67.7
Baseline macro F1: 69.0


## Tokenize

In [17]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128, padding="max_length")

train_ds = Dataset.from_pandas(train[["text", "labels"]]).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(val[["text", "labels"]]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test[["text", "labels"]]).map(tokenize, batched=True)

Map:   0%|          | 0/5532 [00:00<?, ? examples/s]

Map:   0%|          | 0/2275 [00:00<?, ? examples/s]

Map:   0%|          | 0/1607 [00:00<?, ? examples/s]

## Train

In [18]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=8,
    problem_type="multi_label_classification",
)

args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=10,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds)
trainer.train()

[transformers] You passed `num_labels=8` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

Epoch,Training Loss,Validation Loss
1,No log,0.137303
2,No log,0.078297
3,0.174391,0.058938
4,0.174391,0.057369
5,0.174391,0.049088
6,0.037456,0.052076
7,0.037456,0.048822
8,0.037456,0.048019
9,0.019989,0.049339
10,0.019989,0.048624


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1730, training_loss=0.06904709159983376, metrics={'train_runtime': 944.5725, 'train_samples_per_second': 58.566, 'train_steps_per_second': 1.832, 'total_flos': 3639021924925440.0, 'train_loss': 0.06904709159983376, 'epoch': 10.0})

## Choose threshold

In [20]:
def get_probs(ds):
    logits = trainer.predict(ds).predictions
    return 1 / (1 + np.exp(-logits))

val_probs = get_probs(val_ds)

thresholds = []
for i in range(8):
    best_t, best_f1 = 0.5, 0
    for t in np.arange(0.1, 0.9, 0.05):
        f1 = f1_score(y_val[:, i], val_probs[:, i] > t)
        if f1 > best_f1:
            best_t, best_f1 = t, f1
    thresholds.append(round(best_t, 2))

thresholds = np.array(thresholds)
pd.Series(thresholds, index=NAMES)

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Limitation of liability    0.65
Unilateral termination     0.65
Unilateral change          0.60
Content removal            0.70
Contract by using          0.85
Choice of law              0.15
Jurisdiction               0.20
Arbitration                0.50
dtype: float64

## Evaluate on test

In [21]:
test_probs = get_probs(test_ds)
bert_pred = (test_probs > thresholds).astype(int)

results = pd.DataFrame({
    "micro F1": [f1_score(y_test, baseline_pred, average="micro"), f1_score(y_test, bert_pred, average="micro")],
    "macro F1": [f1_score(y_test, baseline_pred, average="macro"), f1_score(y_test, bert_pred, average="macro")],
}, index=["Baseline", "Legal-BERT"])

(results * 100).round(1)

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


,micro F1,macro F1
Baseline,67.7,69.0
Legal-BERT,80.9,82.9


In [22]:
print(classification_report(y_test, bert_pred, target_names=NAMES, zero_division=0))

                         precision    recall  f1-score   support

Limitation of liability       0.78      0.74      0.76        38
 Unilateral termination       0.78      0.82      0.79        38
      Unilateral change       0.79      0.68      0.73        38
        Content removal       0.79      0.85      0.81        13
      Contract by using       0.95      0.78      0.86        23
          Choice of law       0.87      1.00      0.93        13
           Jurisdiction       0.94      1.00      0.97        16
            Arbitration       0.64      1.00      0.78         7

              micro avg       0.81      0.81      0.81       186
              macro avg       0.81      0.86      0.83       186
           weighted avg       0.82      0.81      0.81       186
            samples avg       0.09      0.09      0.09       186



## Save model

In [23]:
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(f"{SAVE_DIR}/thresholds.txt", "w") as f:
    f.write(",".join(str(t) for t in thresholds))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Try it

In [24]:
def predict(clauses):
    inputs = tokenizer(clauses, truncation=True, max_length=128, padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = torch.sigmoid(model(**inputs).logits).cpu().numpy()

    results = []
    for clause, p in zip(clauses, probs):
        labels = [NAMES[i] for i in range(8) if p[i] > thresholds[i]]
        results.append({"clause": clause, "is_unfair": len(labels) > 0, "labels": labels})
    return results

examples = [
    "We may terminate your account at any time without notice.",
    "Any dispute shall be resolved by binding arbitration.",
    "These terms are governed by the laws of California.",
    "You can contact our support team by email.",
]

pd.DataFrame(predict(examples))

,clause,is_unfair,labels
0,We may terminate your account at any time with...,True,[Unilateral termination]
1,Any dispute shall be resolved by binding arbit...,True,[Arbitration]
2,These terms are governed by the laws of Califo...,True,[Choice of law]
3,You can contact our support team by email.,False,[]
